In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# Start

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        df = df[df['Non_Standard_Braking'] == 0]
        df = df[df['SV_Error'] == 0]
        df = df[df['BC_BadStart'] == 0]
        df = df[df['First_phase_error'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source
        df['Malfunction'] = 0

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)
    level = [2]
    df_reference = df_reference[df_reference['Sensor'].isin(level)]
    # Binary label from malfunction code
    leakage_codes = ['H']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Manual_brake',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0
    
    # Add BC_ID and WV_ID to align with Monorail schema
    df_reference['BC_ID'] = 'SD'
    df_reference['WV_ID'] = 'SD'

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action',
        'Max_pressure_exp':               'BC_MaxPressure'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Manual_brake': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 1.8 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base, df_reference

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_raw_Dati01.csv',
    'TestBrakefinal_data_raw_Dati06.csv',
    'TestBrakefinal_data_raw_Dati27.csv'
]

[df, df_reference] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

# Explore San Donato data

In [ ]:
# Comparison with the sensors associated with leakages
# level = [2]
# df_reference = df_reference[df_reference['Sensor'].isin(level)]

common_cols = df_reference.columns.intersection(df.columns)

Features = df_reference[common_cols]
Features = Features.drop(columns=['WV_MeanPressure','EmergencyBrake_action'])
Parameters = df_reference[['WV_MeanPressure','EmergencyBrake_action','Brake_mode','Frequency','Sensor']]
Target = df_reference['LeakageLabel']
Target_raw = df_reference['Malfunction']
Features = Features.drop(columns=['BC_ID','WV_ID','Source','Malfunction'])
print(Features.shape)
Features.head(len(Features.columns))

In [ ]:
df_SanDonato = Features.loc[
    (Features['First_phase_first_gradient'] == 0)
].copy()
df_SanDonato

In [ ]:
Parameters.head()

In [ ]:
# Impute missing (with median)
imp = SimpleImputer(strategy="median")
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
#Features = Features.dropna(axis=1)

# Drop columns that are all NaN or constant
Features = Features.loc[:, Features.notna().any()]  # drop all-NaN
const_mask = Features.nunique(dropna=True) <= 1
if const_mask.any():
    Features = Features.loc[:, ~const_mask]

print(Features.shape)
Features.columns

In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold

Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# # Example: drop features with variance below 1e-2 AFTER scaling (optional):
# vt = VarianceThreshold(threshold=1e-2)
# X_var = vt.fit_transform(Xsel)
# kept_mask = vt.get_support()
# kept_features = Xsel.columns[kept_mask]
# Xsel = Xsel[kept_features]


# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
# REDUCED trees for small dataset stability
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=3, 
                            random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False, n_neighbors=3)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings with OPTIMIZED WEIGHTS for 60 samples ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# WEIGHTED OverallRank for small datasets (60 samples)
# MI: 0.50 (highest weight - most reliable for small data)
# ANOVA: 0.30 (second - stable if linear relationships exist)
# RF: 0.20 (lowest - prone to overfitting with 60 samples)
weights = {
    "MI_Rank": 0.70,
    "ANOVA_Rank": 1,
    "RF_Rank": 0.20
}

rank_table["OverallRank"] = (
    rank_table["MI_Rank"] * weights["MI_Rank"] +
    rank_table["ANOVA_Rank"] * weights["ANOVA_Rank"] +
    rank_table["RF_Rank"] * weights["RF_Rank"]
)

# Sort and display top-N
N = 5
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by Weighted OverallRank (lower = better) ===")
print(f"Weights: MI={weights['MI_Rank']}, ANOVA={weights['ANOVA_Rank']}, RF={weights['RF_Rank']}")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

# Explore which Sensor subjected to heaviest loading condition
pneumatically related to central bogie sensor to be used for manual braking prediction

In [ ]:
# SAN DONATO BC & WV IDs is 'SD' as placeholder
# Ensure ID fields are strings
df["BC_ID"] = df["BC_ID"].astype(str)
df["WV_ID"] = df["WV_ID"].astype(str)
df["Source"] = df["Source"].astype(str)

# ============================================
# 2. OPTIONAL: Use only healthy data for load estimation
# ============================================
if "LeakageLabel" in df.columns:
    df_h = df[(df["LeakageLabel"].str.lower() == "healthy") & (df["DataSource"] == 1)].copy()
else:
    df_h = df[df["DataSource"] == 1].copy()

# ============================================
# 3. SAFE CORRELATION FUNCTION
# ============================================
def safe_corr(x, y):
    if len(x) < 3:
        return np.nan
    if np.isclose(np.nanstd(x), 0) or np.isclose(np.nanstd(y), 0):
        return np.nan
    return np.corrcoef(x, y)[0, 1]

# ============================================
# 4. PER-SOURCE ANALYSIS
# ============================================
sources = df_h["Source"].unique()

all_results = {}
TAU_WV = 1.8   # physical load boundary [bar]

for src in sources:
    df_s = df_h[df_h["Source"] == src]

    # Group by BC–WV pairs for this specific wagon
    stats = (
    df_s.groupby(["BC_ID", "WV_ID"])[["WV_MeanPressure", "BC_MaxPressure"]]
    .apply(lambda g: pd.Series({
        "n_samples": len(g),
        "WV_mean": g["WV_MeanPressure"].mean(),
        "WV_std":  g["WV_MeanPressure"].std(),
        "BC_mean": g["BC_MaxPressure"].mean(),
        "BC_max":  g["BC_MaxPressure"].max(),
        "corr_WV_BC": safe_corr(g["WV_MeanPressure"], g["BC_MaxPressure"])
    }))
    .reset_index()
    )  


    # Sort from highest load (WV_mean)
    stats_sorted = stats.sort_values("WV_mean", ascending=False)
    all_results[src] = stats_sorted

    print("\n======================================")
    print(f" LOAD RANKING — KIT {src}")
    print("======================================")
    print(stats_sorted.to_string(index=False))

    # =====================================
    # HEATMAP (WV Mean)
    # =====================================
    pivot = stats.pivot(index="BC_ID", columns="WV_ID", values="WV_mean")

    plt.figure(figsize=(8, 5))
    center_val = df_h["WV_MeanPressure"].mean()
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",   # diverging palette
        center=center_val,
        cbar_kws={"label": "WV_MeanPressure [bar]"}
    )
    

    # =====================================
    # SCATTER PLOTS FOR EACH PAIR
    # =====================================
    for _, row in stats_sorted.iterrows():
        bc = row["BC_ID"]
        wv = row["WV_ID"]

        df_pair = df_s[(df_s["BC_ID"] == bc) & (df_s["WV_ID"] == wv)]
        if len(df_pair) < 10:
            continue
        df_pair = df_pair.copy()
        df_pair["Regime"] = np.where(
            df_pair["WV_MeanPressure"] < TAU_WV, "Unloaded", "Loaded"
        )
        plt.figure(figsize=(9, 4))
        ax = sns.scatterplot(
            data=df_pair,
            x="WV_MeanPressure", y="BC_MaxPressure",
            hue="Regime", palette={"Unloaded": "steelblue", "Loaded": "darkorange"},
            alpha=0.6
        )

        # Overlay regression line (without scatter points)
        sns.regplot(
            data=df_pair,
            x="WV_MeanPressure", y="BC_MaxPressure",
            scatter=False, line_kws={"linewidth": 1, "color": "green"},
            ax=ax
        )
        # Physical load boundary (example value, adjust as needed)    
        # --- Physical load boundary ---
        ax.axvline(
            TAU_WV,
            linestyle="--",
            linewidth=2,
            color="black",
            label=f"Load boundary (WV = {TAU_WV:.1f} bar)"
        )

        ax.legend(loc="best", fontsize=9)
        plt.title(f"KIT {src} — Pair {bc} ↔ {wv}")
        plt.xlabel("WV_MeanPressure [bar]")
        plt.ylabel("BC_MaxPressure [bar]")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

# all_results is a dictionary: {source → ranking table}


# Filter out based on WV_MeanPressure > 2 bar 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
# SAN DONATO BC & WV IDs is 'SD' as placeholder
# Ensure ID fields are strings
df["BC_ID"] = df["BC_ID"].astype(str)
df["WV_ID"] = df["WV_ID"].astype(str)
df["Source"] = df["Source"].astype(str)
TAU_WV = 1.8   # physical load boundary [bar]

# ============================================
# 2. OPTIONAL: Use only healthy data for load estimation
# ============================================
if "LeakageLabel" in df.columns:
    df_h = df[(df["LeakageLabel"].str.lower() == "healthy") & (df["DataSource"] == 1)].copy()
else:
    df_h = df[df["DataSource"] == 1].copy()

# ============================================
# WV >= 1.7 bar (loaded regime)
# ============================================
df_h = df_h[df_h["WV_MeanPressure"] >= TAU_WV].copy()

print(f"Total samples after filtering WV >= 2: {len(df_h)}")

# ============================================
# 2. SAFE CORRELATION
# ============================================
def safe_corr(x, y):
    if len(x) < 3:
        return np.nan
    if np.isclose(np.nanstd(x), 0) or np.isclose(np.nanstd(y), 0):
        return np.nan
    return np.corrcoef(x, y)[0, 1]

# ============================================
# 3. PER-SOURCE ANALYSIS (NO WARNING)
# ============================================
sources = df_h["Source"].unique()
all_results = {}

for src in sources:
    df_s = df_h[df_h["Source"] == src].copy()

    if df_s.empty:
        print(f"\nSource {src}: no WV >=2 data available.")
        continue

    # Select only relevant columns BEFORE apply → no DeprecationWarning
    grouped = df_s.groupby(["BC_ID", "WV_ID"])

    stats = grouped.apply(
        lambda g: pd.Series({
            "n_samples": len(g),
            "WV_mean": g["WV_MeanPressure"].mean(),
            "WV_std":  g["WV_MeanPressure"].std(),
            "BC_mean": g["BC_MaxPressure"].mean(),
            "BC_max":  g["BC_MaxPressure"].max(),
            "corr_WV_BC": safe_corr(g["WV_MeanPressure"], g["BC_MaxPressure"])
        }),
        include_groups=False  # future-proof
    ).reset_index()

    # Sort highest → lowest load
    stats_sorted = stats.sort_values("WV_mean", ascending=False)
    all_results[src] = stats_sorted

    print("\n======================================")
    print(f" LOAD RANKING — SOURCE {src} (WV >= 2 only)")
    print("======================================")
    print(stats_sorted.to_string(index=False))

    # =====================================
    # HEATMAP (WV Mean)
    # =====================================
    pivot = stats.pivot(index="BC_ID", columns="WV_ID", values="WV_mean")

    plt.figure(figsize=(8, 5))
    center_val = df_h["WV_MeanPressure"].mean()
    sns.heatmap(
        pivot,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",   # diverging palette
        center=center_val,
        cbar_kws={"label": "WV_MeanPressure [bar]"}
    )



# Filter the Monorail Data

In [ ]:
# From df_h select only the row with WV_ID that is SD, 0x43, 0x9d, 0xfc
# Define WV_ID values correlated to central bogie and San Donato
allowed_wv_ids = ["SD", "0x43", "0x9d", "0xfc"]

# Filter df_h to only those WV_IDs
df_filtered = df[df["WV_ID"].isin(allowed_wv_ids)].copy()

print(f"Total samples after filtering by central bogie (including San Donato): {(df_filtered.shape)}")

# Visualization Scatter

In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot First_phase_mean_curvature against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'First_phase_mean_curvature', 'label', and selected features.
    features : list of str
        List of feature column names to plot against First_phase_mean_curvature.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (First_phase_mean_curvature).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'First_phase_mean_curvature'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'First_phase_mean_curvature'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("First_phase_mean_curvature")
        plt.ylabel(feature)
        plt.title(f"First_phase_mean_curvature vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:

selected_features = ["First_phase_half_time_ratio", "First_phase_power","First_phase_timing_half","First_phase_first_gradient"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=None)


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot First_phase_mean_curvature against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'First_phase_mean_curvature', 'label', and selected features.
    features : list of str
        List of feature column names to plot against First_phase_mean_curvature.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (First_phase_mean_curvature).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'First_phase_mean_curvature'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'First_phase_mean_curvature'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("First_phase_mean_curvature")
        plt.ylabel(feature)
        plt.title(f"First_phase_mean_curvature vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:

selected_features = ["First_phase_half_time_ratio", "First_phase_power"]
df_selected = df_filtered[df_filtered["EmergencyBrake_action"] == 1]
# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_selected, selected_features, x_limits=None)


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot First_phase_mean_curvature against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'First_phase_mean_curvature', 'label', and selected features.
    features : list of str
        List of feature column names to plot against First_phase_mean_curvature.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (First_phase_mean_curvature).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'First_phase_mean_curvature'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'First_phase_mean_curvature'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("First_phase_mean_curvature")
        plt.ylabel(feature)
        plt.title(f"First_phase_mean_curvature vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:

selected_features = ["First_phase_half_time_ratio", "First_phase_power"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=None)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_plot = df_filtered.copy()
df_plot['y_jitter'] = rng.normal(0, 0.02, size=len(df_plot))
# df_plot = df_filtered[(df_filtered['Max_pressure_pipe'] < 0.8) & (df_filtered['WV_bin'] == 1)].copy()
# df_plot = df_filtered[(df_filtered['EmergencyBrake_action'] == 1)].copy()
df_plot = df_plot[(df_plot['WV_bin'] == 2)].copy()
# df_plot = df_filtered[(df_filtered['DataSource'] == 0) & (df_filtered['WV_bin'] == 1)].copy()
# df_plot = df_filtered[(df_filtered['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. DISTINCT palettes for left vs right
# ------------------------------------------------------------

# LEFT: Kit Source → use a soft pastel palette
palette_source = sns.color_palette("pastel", n_colors=len(df_plot['Source'].unique()))

# RIGHT: Label → use a bold, high-contrast palette
label_palette = {
    0: (0.1, 0.2, 0.9, 0.9),   # strong blue
    1: (0.9, 0.2, 0.1, 0.9),   # strong red
}
# or: label_palette = {0: sns.color_palette("deep")[0], 1: sns.color_palette("deep")[3]}

# Map categories to colors
source_codes = df_plot['Source'].astype('category').cat.codes
label_codes  = df_plot['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]
# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_plot['First_phase_mean_curvature'],
    df_plot['y_jitter'],
    c=colors_source,   # from palette_source
    alpha=0.7
)
axes[0].set_title("First_phase_mean_curvature by Kit Source")
axes[0].set_xlabel("First_phase_mean_curvature")
axes[0].set_yticks([])

# Legend for Source
sources = df_plot['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")



# ---------------- Right: coloured by Label ----------------
axes[1].scatter(
    df_plot['First_phase_mean_curvature'],
    df_plot['y_jitter'],
    c=colors_label,    # from label_palette
    alpha=0.7
)
axes[1].set_title("First_phase_mean_curvature by Label")
axes[1].set_xlabel("First_phase_mean_curvature")
axes[1].set_yticks([])

# Legend for Label
labels = df_plot['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Manual Brake Activation", loc="upper right")

# axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_plot = df_filtered.copy()
df_plot['y_jitter'] = rng.normal(0, 0.02, size=len(df_plot))
# df_plot = df_filtered[(df_filtered['EmergencyBrake_action'] == 1)].copy()
# df_plot = df_filtered[(df_filtered['DataSource'] == 0)].copy()
# df_plot = df_filtered[(df_filtered['DataSource'] == 0) & (df_filtered['WV_bin'] == 1)].copy()
df_plot = df_plot[(df_plot['WV_bin'] == 2)].copy()


# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_plot['First_phase_half_time_ratio'],
    df_plot['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("First_phase_half_time_ratio by Kit Source")
axes[0].set_xlabel("First_phase_half_time_ratio")
axes[0].set_yticks([])

# Source legend
sources = df_plot['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper left")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_plot['First_phase_half_time_ratio'],
    df_plot['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("First_phase_half_time_ratio by Label")
axes[1].set_xlabel("First_phase_half_time_ratio")
axes[1].set_yticks([])

# Label legend
labels = df_plot['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Manual Brake Activation", loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
# df_plot = df_filtered[(df_filtered['EmergencyBrake_action'] == 1)].copy()
# df_plot = df_filtered[(df_filtered['DataSource'] == 0) & (df_filtered['WV_bin'] == 1)].copy()
df_plot = df_plot[(df_plot['WV_bin'] == 2)].copy()

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_plot['First_phase_power'],
    df_plot['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("First_phase_power by Kit Source")
axes[0].set_xlabel("First_phase_power")
axes[0].set_yticks([])

# Source legend
sources = df_plot['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_plot['First_phase_power'],
    df_plot['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("First_phase_power by Label")
axes[1].set_xlabel("First_phase_power")
axes[1].set_yticks([])

# Label legend
labels = df_plot['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Manual Brake Activation", loc="upper right")
# axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
# df_plot = df_filtered[(df_filtered['EmergencyBrake_action'] == 1)].copy()
# df_plot = df_filtered[(df_filtered['DataSource'] == 0) & (df_filtered['WV_bin'] == 1)].copy()
df_plot = df_plot[(df_plot['WV_bin'] == 2)].copy()

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_plot['First_phase_first_gradient'],
    df_plot['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("First_phase_first_gradient by Kit Source")
axes[0].set_xlabel("First_phase_first_gradient")
axes[0].set_yticks([])
axes[0].set_xlim(-1, 2)
# Source legend
sources = df_plot['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper left")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_plot['First_phase_first_gradient'],
    df_plot['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("First_phase_first_gradient by Label")
axes[1].set_xlabel("First_phase_first_gradient")
axes[1].set_yticks([])

# Label legend
labels = df_plot['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Manual Brake Activation", loc="upper left")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
y_test = df_filtered[df_filtered['label']==1]

y_test.head()

In [ ]:
X_healthy = df_filtered[df_filtered['label']==0]
print(X_healthy.shape)

## Scaling

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
## matplotlib inline
matplotlib.style.use('fivethirtyeight')

df_plot = df_filtered.copy()
selected_features = ["First_phase_mean_curvature", "First_phase_half_time_ratio", "First_phase_power"]
x = X_healthy[selected_features]
robust_df = pd.DataFrame(preprocessing.RobustScaler().fit_transform(x), 
                         columns=selected_features)
standard_df = pd.DataFrame(preprocessing.StandardScaler().fit_transform(x), 
                           columns=selected_features)
minmax_df = pd.DataFrame(preprocessing.MinMaxScaler().fit_transform(x), 
                         columns=selected_features)

fig, axes = plt.subplots(ncols=4, figsize=(20, 5))
datasets = [x, robust_df, standard_df, minmax_df]
titles = ['Before Scaling', 'Robust Scaling', 'Standard Scaling', 'Min-Max Scaling']
colors = ['r', 'b', 'g', 'cyan', 'orange', 'purple']  # extend for more features

for ax, df_plot, title in zip(axes, datasets, titles):
    ax.set_title(title)
    for i, feature in enumerate(df_plot.columns):
        sns.kdeplot(df_plot[feature], ax=ax, color=colors[i % len(colors)], label=feature)
    ax.legend()
plt.show()

## Preprocessing

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

def preprocess_data_algorithm(df, features, holdout_frac=0.2, random_state=42):
    """
    Baseline setup:
      - Filter to a consistent WV regime and physically valid first-phase gradient
      - Use DataSource==1 as the healthy baseline pool
      - Split baseline pool into:
          * fit set (used for training / threshold fitting)
          * holdout set (healthy, appended to test)
      - Test set contains original DataSource==0 samples + appended healthy holdout

    Returns
    -------
    X_train          : features from baseline fit set (subset of DataSource==1)
    X_test           : features from (DataSource==0) + (baseline holdout)
    y_train          : zeros (healthy baseline)
    y_test           : labels for the extended test set
    test_orig_index  : original df indices for X_test rows
    """

    df = df.copy()
    df["orig_index"] = df.index

    # --------------------------------------------------
    # 1) Regime + physical consistency filter
    # --------------------------------------------------
    df_filt = df[
        df["WV_bin"].between(1, 2)
    ].copy()

    # --------------------------------------------------
    # 2) Split by DataSource
    # --------------------------------------------------
    df_train_pool = df_filt[df_filt["DataSource"] == 1].copy()
    df_test = df_filt[df_filt["DataSource"] == 0].copy()

    # --------------------------------------------------
    # 3) Train/holdout split within baseline pool
    # --------------------------------------------------
    if holdout_frac and (0 < holdout_frac < 1) and len(df_train_pool) >= 2:
        df_train_fit, df_train_holdout = train_test_split(
            df_train_pool,
            test_size=holdout_frac,
            shuffle=True,
            random_state=random_state
        )
    else:
        df_train_fit = df_train_pool
        df_train_holdout = df_train_pool.iloc[0:0].copy()  # empty

    # --------------------------------------------------
    # TRAIN (healthy-only)
    # --------------------------------------------------
    X_train = df_train_fit[features].reset_index(drop=True)
    y_train = pd.Series(0, index=X_train.index, name="label")

    # --------------------------------------------------
    # TEST (original DataSource==0 + appended healthy holdout)
    # --------------------------------------------------
    df_test_ext = pd.concat([df_test, df_train_holdout], ignore_index=True)

    X_test = df_test_ext[features].reset_index(drop=True)
    y_test = df_test_ext["label"].reset_index(drop=True)
    test_orig_index = df_test_ext["orig_index"].reset_index(drop=True)
    test_meta = df_test_ext[[
        "orig_index",
        "DataSource",
        "label"
    ]].reset_index(drop=True)
    # --------------------------------------------------
    # BASIC COUNTS
    # --------------------------------------------------
    print(f"Total samples (before WV filter): {len(df)}")
    print(f"Filtered samples (WV_bin in [1, 2] & grad>=0): {len(df_filt)}")
    print(f"  Healthy set (DataSource==1): {len(df_train_pool)}")
    print(f"    Train fit: {len(df_train_fit)}")
    print(f"    Holdout→Test: {len(df_train_holdout)}")
    print(f"  Test pool (DataSource==0): {len(df_test)}")

    print(f"\nTRAIN: {len(X_train)} (healthy baseline fit)")
    print(f"TEST : {len(X_test)} "
          f"(healthy={int((y_test==0).sum())}, unhealthy={int((y_test==1).sum())})")

    # --------------------------------------------------
    # NaN DIAGNOSTICS (per feature)
    # --------------------------------------------------
    nan_report = pd.DataFrame({
        "NaN_train": X_train.isna().sum(),
        "NaN_test":  X_test.isna().sum()
    })

    print("\nNaN count per feature:")
    print(nan_report)

    return X_train, X_test, y_train, y_test, test_orig_index, test_meta


In [ ]:
[df, df_reference] = load_data(model_path, monorail_paths)
df['Malfunction'] = df['Malfunction'].astype(str)
print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail
df["orig_index"] = df.index 

In [ ]:
# Define allowed WV_IDs correlated to central bogie and San Donato
allowed_wv_ids = ["SD", "0x43", "0x9d", "0xfc"]

# Filter df_h by both WV_ID and WV_bin (2 or 3)
df_filtered = df[
    (df["WV_ID"].isin(allowed_wv_ids)) &
    (df["WV_bin"].isin([2, 3]))
].copy()

print(f"Total samples after filtering by central bogie (including San Donato, WV_bin 2 & 3): {df_filtered.shape}")

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
features = [
    "First_phase_mean_curvature",
    "First_phase_power",
    "First_phase_half_time_ratio",
    "First_phase_first_gradient",
]

X_train, X_test, y_train, y_test, test_orig_index, test_meta = preprocess_data_algorithm(
    df_filtered, features, holdout_frac=0.2, random_state=42
)

In [ ]:
import numpy as np

# positions of unhealthy samples *inside TEST*
fault_idx = np.where(y_test == 1)[0]
print("Faulty samples in TEST positions:", fault_idx)

# original row indices in the full df_filtered
orig_fault_idx = test_orig_index.iloc[fault_idx].values
print("Original dataframe indices:", orig_fault_idx)

# inspect those rows in the original dataframe
df_filtered.loc[orig_fault_idx]


In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler

def scale_features(X_train, X_test):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)

    return X_train_scaled, X_test_scaled, scaler, imputer

# Align y_train_healthy with X_train_healthy

[X_train_scaled, X_test_scaled, scaler, imputer] = scale_features(X_train, X_test)

# Exploring with PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# -------------------------------------------------
# 1. Fit PCA on the ORIGINAL data with all components
# -------------------------------------------------
pca_full = PCA(n_components=len(features))   # since you had 5 features
pca_full.fit(X_train_scaled)

# -------------------------------------------------
# 2. Scree plot
# -------------------------------------------------
explained_var = pca_full.explained_variance_ratio_ *100

plt.figure(figsize=(6, 4))
plt.bar(range(1, len(explained_var) + 1), explained_var, color='royalblue')
plt.title("Scree Plot – Explained Variance by Principal Components")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.xticks(range(1, len(explained_var) + 1))
plt.grid(True, linestyle='--', alpha=0.6)

# Add cumulative variance line for clarity
plt.plot(range(1, len(explained_var) + 1), np.cumsum(explained_var), 'o--', color='darkorange', label="Cumulative")
plt.legend()
plt.tight_layout()
plt.show()

print("Explained variance ratio:", explained_var)
print("Cumulative variance:", np.cumsum(explained_var))

In [ ]:
import pandas as pd

# Use the fitted PCA object (pca_full or pca depending on your workflow)
n_components = pca_full.n_components_   # number of PCs actually fitted

# Generate column names PC1, PC2, ..., PCn
pc_names = [f'PC{i+1}' for i in range(n_components)]

# Build loadings DataFrame
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=pc_names,
    index=features
)

print(loadings)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for f in features:
    print(f"{f}: min = {X_train[f].min():.3f}, max = {X_train[f].max():.3f}, skew = {X_train[f].skew():.2f}")


for f in features:
    sns.histplot(X_train[f], kde=True)
    plt.title(f"Histogram of {f}")
    plt.show()


# Statistical Analysis - Mahalanobis Distance

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
features = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio",
]

X_train, X_test, y_train, y_test, test_orig_index, test_meta = preprocess_data_algorithm(
    df_filtered, features
)

# --- create masks ---
mask_tr = X_train[features].notna().all(axis=1)
mask_te = X_test[features].notna().all(axis=1)

# --- apply masks ---
X_train = X_train.loc[mask_tr].copy()
y_train = y_train.loc[mask_tr].reset_index(drop=True)

X_test = X_test.loc[mask_te].copy()
y_test = y_test.loc[mask_te].reset_index(drop=True)
test_meta = test_meta.loc[mask_te].reset_index(drop=True)

print(f"After dropping NaNs, TRAIN shape: {X_train.shape}, TEST shape: {X_test.shape}")

[X_train_scaled, X_test_scaled, scaler, imputer] = scale_features(X_train, X_test)

## Compute Mahalanobis Distance

In [ ]:
import numpy as np

# Convert to numpy array
X_train_np = np.asarray(X_train, dtype=float)

# Mean vector (d-dimensional)
mu = X_train_np.mean(axis=0)

# Compute Mahalanobis distances for training data
# Centered data
diff_train = X_train_np - mu
cov = np.cov(diff_train, rowvar=False) # Covariance matrix (d x d)
cov_inv = np.linalg.pinv(cov) # Pseudo-inverse of covariance (more stable than inv)
temp_train = diff_train @ cov_inv   # temp_i = (x_i - mu) S^-1

# Each Mahalanobis squared = sum over j: temp_ij * diff_ij
MD_sq_train = np.sum(temp_train * diff_train, axis=1)
d_train = np.sqrt(MD_sq_train) # MD distances

print("Train distances (min, median, max):",
      np.min(d_train), np.median(d_train), np.max(d_train))
print("mean of each features:", mu)
print("covariance shape:", cov.shape)
print("covariance matrix:\n", cov)

In [ ]:
from scipy.stats import chi2

alpha = 0.95  # confidence level
T_emp = np.quantile(d_train, alpha)
print("Empirical threshold (quantile):", T_emp)


d = X_train_np.shape[1]  # number of features (2 or 3)
T_chi = np.sqrt(chi2.ppf(alpha, df=d))
print("Chi-square-based threshold:", T_chi)

T = T_emp
print(f"Using threshold T = {T:.4f} (empirical {alpha*100:.1f}%-quantile)")


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def plot_mahalanobis_ellipse(mu, cov, threshold, ax=None, **kwargs):
    """
    Plot an ellipse representing Mahalanobis distance threshold.

    Parameters
    ----------
    mu : array-like, shape (2,)
        Mean of the distribution.
    cov : array-like, shape (2, 2)
        Covariance matrix.
    threshold : float
        Mahalanobis distance threshold (not squared).
    ax : matplotlib Axes (optional)
        Axes to plot on.
    kwargs : passed to Ellipse
    """
    if ax is None:
        fig, ax = plt.subplots()

    # Eigen-decomposition for axis length and rotation
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    angle = np.degrees(np.arctan2(*eigvecs[:, 0][::-1]))

    # Width and height of ellipse: 2 * axis_length * threshold
    width, height = 2 * threshold * np.sqrt(eigvals)

    ell = Ellipse(xy=mu, width=width, height=height, angle=angle,
                  edgecolor='red', facecolor='none', linestyle='--', linewidth=2, **kwargs)
    ax.add_patch(ell)

    return ax

fig, ax = plt.subplots(figsize=(8, 6))

# Scatter plot of healthy data
ax.scatter(X_train_np[:, 0], X_train_np[:, 1], c='gray', s=20, label='Train (healthy)')

# Plot Mahalanobis ellipse at threshold T
plot_mahalanobis_ellipse(mu, cov, threshold=T, ax=ax)

ax.set_title("Mahalanobis Distance Ellipse (Chi² {:.0f}%)".format(alpha * 100))
ax.set_xlabel(features[0])
ax.set_ylabel(features[1])
ax.legend()
plt.grid(True)
plt.axis('equal')
plt.show()


In [ ]:
# Convert X_test to numpy
X_test_np = np.asarray(X_test, dtype=float)

# Center and project through Σ^-1
diff_test = X_test_np - mu
temp_test = diff_test @ cov_inv

MD_sq_test = np.sum(temp_test * diff_test, axis=1)
d_test = np.sqrt(MD_sq_test)

print("Test distances (first 10):", d_test[:10])


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# y_test is a pandas Series: 0=healthy, 1=fault
y_pred = (d_test > T).astype(int)  # numpy array: 1 = anomaly, 0 = normal

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion matrix (rows=true, cols=pred):\n", cm)

# Unpack if you like
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Fault"]))


In [ ]:
# Boolean masks
mask_fault   = (y_test == 1)
mask_healthy = (y_test == 0)

d_fault   = d_test[mask_fault]
d_healthy = d_test[mask_healthy]

print("Fault distances:", d_fault)
print("Threshold     :", T)
print("Healthy distances (min, median, max):",
      np.min(d_healthy), np.median(d_healthy), np.max(d_healthy))

print("Are faults flagged as anomalies?:", (d_fault > T))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def add_md_ellipse(ax, mu_2d, cov_2d, T, label, edgecolor="k", lw=2, zorder=5):
    """
    Add ellipse for (x-mu)^T Sigma^{-1} (x-mu) = T^2
    Returns (xmin, xmax, ymin, ymax) bounds of the ellipse.
    """
    mu_2d = np.asarray(mu_2d, dtype=float).reshape(2,)
    cov_2d = np.asarray(cov_2d, dtype=float).reshape(2,2)

    # Eigen decomposition (cov must be PSD; clip tiny negatives)
    eigvals, eigvecs = np.linalg.eigh(cov_2d)
    eigvals = np.maximum(eigvals, 0.0)
    order = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))

    c = float(T)**2
    # Full widths used by matplotlib Ellipse
    width  = 2.0 * np.sqrt(c * eigvals[0])
    height = 2.0 * np.sqrt(c * eigvals[1])

    ell = Ellipse(
        xy=mu_2d, width=width, height=height, angle=angle,
        fill=False, edgecolor=edgecolor, linewidth=lw,
        label=label, zorder=zorder
    )
    ax.add_patch(ell)

    # Conservative bounds (axis-aligned) for autoscaling
    xmin, xmax = mu_2d[0] - width/2,  mu_2d[0] + width/2
    ymin, ymax = mu_2d[1] - height/2, mu_2d[1] + height/2
    return xmin, xmax, ymin, ymax


def plot_md_ellipses_2d(
    X_train_2d, X_test_2d, y_test, mu_2d, cov_2d,
    T_emp, T_chi, alpha=0.95,
    title="MD ellipses in feature space"
):
    X_train_2d = np.asarray(X_train_2d, dtype=float)
    X_test_2d  = np.asarray(X_test_2d, dtype=float)
    y_test     = np.asarray(y_test).astype(int)

    fig, ax = plt.subplots(figsize=(8,6))

    ax.scatter(X_train_2d[:,0], X_train_2d[:,1], s=12, alpha=0.35, label="Train")
    ax.scatter(X_test_2d[y_test==0,0], X_test_2d[y_test==0,1], s=18, alpha=0.9, label="Test (Healthy)")
    ax.scatter(X_test_2d[y_test==1,0], X_test_2d[y_test==1,1], s=28, alpha=0.9, marker="X", label="Test (Fault)")
    ax.scatter(mu_2d[0], mu_2d[1], s=140, marker="*", label="Centroid (mu)", zorder=6)

    # Add ellipses and collect bounds
    bounds = []
    bounds.append(add_md_ellipse(ax, mu_2d, cov_2d, T_emp, f"Empirical (q={alpha:.2f})", edgecolor="tab:blue"))
    bounds.append(add_md_ellipse(ax, mu_2d, cov_2d, T_chi, f"Chi-square (q={alpha:.2f})", edgecolor="tab:orange"))

    # Expand axes to include ellipse bounds + a little margin
    xmin = min(b[0] for b in bounds); xmax = max(b[1] for b in bounds)
    ymin = min(b[2] for b in bounds); ymax = max(b[3] for b in bounds)

    # Also include scatter ranges (in case ellipse is small)
    xmin = min(xmin, np.min(X_train_2d[:,0]), np.min(X_test_2d[:,0]))
    xmax = max(xmax, np.max(X_train_2d[:,0]), np.max(X_test_2d[:,0]))
    ymin = min(ymin, np.min(X_train_2d[:,1]), np.min(X_test_2d[:,1]))
    ymax = max(ymax, np.max(X_train_2d[:,1]), np.max(X_test_2d[:,1]))

    mx = 0.08*(xmax-xmin + 1e-9)
    my = 0.08*(ymax-ymin + 1e-9)
    ax.set_xlim(xmin-mx, xmax+mx)
    ax.set_ylim(ymin-my, ymax+my)

    ax.set_title(title)
    ax.set_xlabel("FMPC")
    ax.set_ylabel("FPHTR")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
# If you already built X_train_np with the imputer, use the same for plotting:
X_train_2d = X_train_np[:, :2]

# IMPORTANT: apply the SAME imputer to test before plotting (if you used one on train)
X_test_imp = imputer.transform(X_test)
X_test_2d = np.asarray(X_test_imp, dtype=float)[:, :2]

mu_2d = mu[:2]
cov_2d = cov[:2, :2]

plot_md_ellipses_2d(
    X_train_2d, X_test_2d, y_test,
    mu_2d, cov_2d,
    T_emp=T_emp, T_chi=T_chi, alpha=alpha,
    title="MD ellipses in feature space"
)


# Log transformed Mahalanobis Distance

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
features = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio",
]

X_train, X_test, y_train, y_test, test_orig_index, test_meta = preprocess_data_algorithm(
    df_filtered, features
)
X_train = X_train.abs()
X_test = X_test.abs()
# --- create masks ---
mask_tr = X_train[features].notna().all(axis=1)
mask_te = X_test[features].notna().all(axis=1)

# --- apply masks ---
X_train = X_train.loc[mask_tr].copy()
y_train = y_train.loc[mask_tr].reset_index(drop=True)

X_test = X_test.loc[mask_te].copy()
y_test = y_test.loc[mask_te].reset_index(drop=True)

print(f"After dropping NaNs, TRAIN shape: {X_train.shape}, TEST shape: {X_test.shape}")
cap = 2.0  # domain-informed upper limit for curvature
# Log transform the feature to reduce skewness
X_train_log = X_train.copy()
X_test_log = X_test.copy()
X_train_log["First_phase_mean_curvature"] = np.log1p(np.minimum(X_train_log["First_phase_mean_curvature"], cap))
X_test_log["First_phase_mean_curvature"] = np.log1p(np.minimum(X_test_log["First_phase_mean_curvature"], cap))

[X_train_scaled, X_test_scaled, scaler, imputer] = scale_features(X_train_log, X_test_log)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(X_train["First_phase_mean_curvature"], bins=30, color='gray')
ax[0].set_title("Original curvature")

ax[1].hist(X_train_log["First_phase_mean_curvature"], bins=30, color='green')
ax[1].set_title("Transformed curvature (log1p, capped)")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(X_test["First_phase_mean_curvature"], bins=30, color='gray')
ax[0].set_title("Original curvature")

ax[1].hist(X_test_log["First_phase_mean_curvature"], bins=30, color='green')
ax[1].set_title("Transformed curvature (log1p, capped)")

plt.tight_layout()
plt.show()

## Compute Mahalanobis Distance

In [ ]:
import numpy as np

# Imputed training data
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_scaled) # use scaled data

# Convert to numpy array
X_train_np = np.asarray(X_train_imp, dtype=float)

# Mean vector (d-dimensional)
mu = X_train_np.mean(axis=0)

# Compute Mahalanobis distances for training data
# Centered data
diff_train = X_train_np - mu
cov = np.cov(diff_train, rowvar=False) # Covariance matrix (d x d)
cov_inv = np.linalg.pinv(cov) # Pseudo-inverse of covariance (more stable than inv)
temp_train = diff_train @ cov_inv   # temp_i = (x_i - mu) S^-1

# Each Mahalanobis squared = sum over j: temp_ij * diff_ij
MD_sq_train = np.sum(temp_train * diff_train, axis=1)
d_train = np.sqrt(MD_sq_train) # MD distances

print("Train distances (min, median, max):",
      np.min(d_train), np.median(d_train), np.max(d_train))
print("mean of each features:", mu)
print("covariance shape:", cov.shape)
print("covariance matrix:\n", cov)

In [ ]:
from scipy.stats import chi2

alpha = 0.95  # confidence level
T_emp = np.quantile(d_train, alpha)
print("Empirical threshold (quantile):", T_emp)


d = X_train_np.shape[1]  # number of features (2 or 3)
T_chi = np.sqrt(chi2.ppf(alpha, df=d))
print("Chi-square-based threshold:", T_chi)

T = T_emp
print(f"Using threshold T = {T:.4f} (empirical {alpha*100:.1f}%-quantile)")


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Convert X_test to numpy
X_test_np = np.asarray(X_test_scaled, dtype=float)

# Center and project through Σ^-1
diff_test = X_test_np - mu
temp_test = diff_test @ cov_inv

MD_sq_test = np.sum(temp_test * diff_test, axis=1)
d_test = np.sqrt(MD_sq_test)

print("Test distances (first 10):", d_test[:10])


# y_test is a pandas Series: 0=healthy, 1=fault
y_pred = (d_test > T).astype(int)  # numpy array: 1 = anomaly, 0 = normal

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print("Confusion matrix (rows=true, cols=pred):\n", cm)

# Unpack if you like
if cm.shape == (2, 2):
    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Fault"]))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

plt.figure(figsize=(8, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred: Healthy", "Pred: Fault"],
    yticklabels=["True: Healthy", "True: Fault"]
)
plt.title("CM: Manual Brake Activation Detection")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()

In [ ]:
# Boolean masks
mask_fault   = (y_test == 1)
mask_healthy = (y_test == 0)

d_fault   = d_test[mask_fault]
d_healthy = d_test[mask_healthy]

print("Fault distances:", d_fault)
print("Threshold     :", T)
print("Healthy distances (min, median, max):",
      np.min(d_healthy), np.median(d_healthy), np.max(d_healthy))

print("Are faults flagged as anomalies?:", (d_fault > T))


In [ ]:
# If you already built X_train_np with the imputer, use the same for plotting:
X_train_2d = X_train_np[:, :2]

# IMPORTANT: apply the SAME imputer to test before plotting (if you used one on train)
X_test_imp = imputer.transform(X_test_scaled)
X_test_2d = np.asarray(X_test_imp, dtype=float)[:, :2]

mu_2d = mu[:2]
cov_2d = cov[:2, :2]

plot_md_ellipses_2d(
    X_train_2d, X_test_2d, y_test,
    mu_2d, cov_2d,
    T_emp=T_emp, T_chi=T_chi, alpha=alpha,
    title="MD ellipses in feature space"
)


# Statistical Threshold with 95% or 92.5% Confidence interval

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING AND FEATURE SELECTION
# ============================================================================
features = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio",
]

X_train, X_test, y_train, y_test, test_orig_index, test_meta = preprocess_data_algorithm(
    df_filtered, features
)
X_train = X_train.abs()  # ensure non-negativity 
X_test = X_test.abs()
# --- create masks ---
mask_tr = X_train[features].notna().all(axis=1)
mask_te = X_test[features].notna().all(axis=1)

# --- apply masks ---
X_train = X_train.loc[mask_tr].copy()
y_train = y_train.loc[mask_tr].reset_index(drop=True)

X_test = X_test.loc[mask_te].copy()
y_test = y_test.loc[mask_te].reset_index(drop=True)
test_meta = test_meta.loc[mask_te].reset_index(drop=True)

print(f"After dropping NaNs, TRAIN shape: {X_train.shape}, TEST shape: {X_test.shape}")

# Imputed training data
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train) # use scaled data

## Quantiles

In [ ]:
import numpy as np
import pandas as pd

from sklearn.covariance import MinCovDet
from scipy.stats import chi2

# -----------------------------
# 0) Rules + quantiles
# -----------------------------
FEATURE_RULES = {
    "First_phase_mean_curvature": "lower",
    "First_phase_half_time_ratio": "upper",
}

Q_LOW  = 0.05 # rule for FPMC
Q_HIGH = 0.95 # rule for FPHTR

# -----------------------------
# 0.5) pick healthy subset from training
# -----------------------------
# Assumption: y_train is 0=healthy, 1=faulty
Xtr = X_train.copy()
Xte = X_test.copy()

Xtr_healthy = Xtr.loc[np.asarray(y_train) == 0, list(FEATURE_RULES.keys())].copy()

# numeric + drop rows with missing any used feature (for multivariate distance)
Xtr_healthy = Xtr_healthy.apply(pd.to_numeric, errors="coerce").dropna(axis=0, how="any")

# -----------------------------
# 0.6) robust multivariate outlier removal (MCD)
# -----------------------------
# Fit robust covariance on healthy
mcd = MinCovDet(support_fraction=0.75, random_state=42).fit(Xtr_healthy.values)

# robust squared Mahalanobis distances
md2 = mcd.mahalanobis(Xtr_healthy.values)

# chi-square cutoff for p features
p = Xtr_healthy.shape[1]
alpha = 0.925
cut = chi2.ppf(alpha, df=p)

mask_inlier = md2 <= cut
Xtr_healthy_clean = Xtr_healthy.loc[mask_inlier].copy()

print(f"Healthy before: {len(Xtr_healthy)} | after outlier filter: {len(Xtr_healthy_clean)}")

# -----------------------------
# 1) Fit thresholds on cleaned healthy
# -----------------------------
thr_rows = []
for feat, rule in FEATURE_RULES.items():
    x = pd.to_numeric(Xtr_healthy_clean[feat], errors="coerce").dropna()

    ql = np.nan
    qh = np.nan
    if len(x) > 0:
        if rule in ("lower", "two"):
            ql = np.nanquantile(x, Q_LOW)
        if rule in ("upper", "two"):
            qh = np.nanquantile(x, Q_HIGH)

    thr_rows.append({"feature": feat, "rule": rule, "q_low": ql, "q_high": qh, "n_fit": len(x)})

thr = pd.DataFrame(thr_rows)
thr_idx = thr.set_index("feature")

# -----------------------------
# 2) Apply thresholds to X_test
# -----------------------------
Xte_scored = Xte.copy()
Xte_scored["anomaly_score"] = 0


Xte_scored["Source"] = test_meta["DataSource"].to_numpy()
Xte_scored["label"]  = test_meta["label"].to_numpy()


for feat, rule in FEATURE_RULES.items():
    vcol = f"viol_{feat}"
    Xte_scored[vcol] = 0

    t = thr_idx.loc[feat]
    x = pd.to_numeric(Xte_scored[feat], errors="coerce")

    if rule == "upper":
        Xte_scored[vcol] = (x > t["q_high"]).astype(int)
    elif rule == "lower":
        Xte_scored[vcol] = (x < t["q_low"]).astype(int)
    else:
        Xte_scored[vcol] = ((x < t["q_low"]) | (x > t["q_high"])).astype(int)

    Xte_scored["anomaly_score"] += Xte_scored[vcol].fillna(0).astype(int)

Xte_scored["anomaly_flag_any"] = (Xte_scored["anomaly_score"] > 0).astype(int)

print(pd.crosstab(y_test, Xte_scored["anomaly_flag_any"], rownames=["y"], colnames=["flag"], normalize="index"))

thresholds = {
    feat: {
        "rule": thr_idx.loc[feat, "rule"],
        "q_low": float(thr_idx.loc[feat, "q_low"]) if pd.notna(thr_idx.loc[feat, "q_low"]) else None,
        "q_high": float(thr_idx.loc[feat, "q_high"]) if pd.notna(thr_idx.loc[feat, "q_high"]) else None,
        "n_fit": int(thr_idx.loc[feat, "n_fit"]),
    }
    for feat in FEATURE_RULES
}
print(thresholds)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

cms = {}

for feat, rule in FEATURE_RULES.items():
    t = thr_idx.loc[feat]
    x = pd.to_numeric(Xte[feat], errors="coerce")

    if rule == "upper":
        pred = (x > t["q_high"])
    elif rule == "lower":
        pred = (x < t["q_low"])
    else:
        pred = (x < t["q_low"]) | (x > t["q_high"])

    y_pred = pred.fillna(False).astype(int).to_numpy()

    y_true = pd.Series(y_test)
    mask = y_true.notna().to_numpy()
    y_true = y_true.to_numpy()[mask].astype(int)
    y_pred = y_pred[mask]

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cms[feat] = cm

    cm_df = pd.DataFrame(
        cm,
        index=["Healthy (0)", "Faulty (1)"],
        columns=["Pred Normal (0)", "Pred Anomaly (1)"]
    )

    print("\n" + "="*70)
    print(f"Single-threshold confusion matrix using: {feat}  (rule={rule})")
    print(f"Threshold used: q_low={t['q_low']:.6g}, q_high={t['q_high']:.6g}")
    print(cm_df)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x_feat = features[0]
y_feat = features[1]

qx = thr.loc[thr["feature"] == x_feat].iloc[0]
qy = thr.loc[thr["feature"] == y_feat].iloc[0]

x_low  = float(qx["q_low"])  if pd.notna(qx.get("q_low", np.nan))  else np.nan
x_high = float(qx["q_high"]) if pd.notna(qx.get("q_high", np.nan)) else np.nan
y_low  = float(qy["q_low"])  if pd.notna(qy.get("q_low", np.nan))  else np.nan
y_high = float(qy["q_high"]) if pd.notna(qy.get("q_high", np.nan)) else np.nan

label_colors = {0: "#4C72B0", 1: "#DD8452"}  # blue, orange

fig, ax = plt.subplots(figsize=(7, 6))

# Train (healthy)
ax.scatter(
    pd.to_numeric(X_train[x_feat], errors="coerce"),
    pd.to_numeric(X_train[y_feat], errors="coerce"),
    color="gray", s=15, alpha=0.25, label="Healthy (train)"
)

# Test grouped by label
for lab in [0, 1]:
    m = (Xte_scored["label"] == lab)
    ax.scatter(
        pd.to_numeric(Xte_scored.loc[m, x_feat], errors="coerce"),
        pd.to_numeric(Xte_scored.loc[m, y_feat], errors="coerce"),
        color=label_colors[lab],
        s=35 if lab == 0 else 45,
        alpha=0.85,
        edgecolor="k",
        linewidth=0.4,
        label=f"Test (label = {lab})"
    )

# Acceptance lines
if np.isfinite(x_low):  ax.axvline(x_low,  color="black", linestyle="--", linewidth=1)
if np.isfinite(x_high): ax.axvline(x_high, color="black", linestyle="--", linewidth=1)
if np.isfinite(y_low):  ax.axhline(y_low,  color="black", linestyle="--", linewidth=1)
if np.isfinite(y_high): ax.axhline(y_high, color="black", linestyle="--", linewidth=1)

ax.set_xlabel(x_feat)
ax.set_ylabel(y_feat)
ax.set_title("Train vs Test scatter (grouped by test label)")
ax.grid(True, alpha=0.3)

# ✅ Legend in FIGURE coords (not axes), so it won't drift
fig.legend(
    loc="upper center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.98)
)

# ✅ Reserve a top band for the legend
fig.tight_layout(rect=[0, 0, 1, 0.92])

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({
    'axes.titlesize': 22,
    'axes.labelsize': 20,
    'xtick.labelsize': 18,
    'ytick.labelsize': 18,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white'
})

features = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio"
]

label_colors = {
    0: "#4C72B0",  # label=0
    1: "#DD8452",  # label=1
}

cats = ["Healthy", "Test"]
xpos = np.array([1, 2], dtype=float)

group_width = 0.60
offset = group_width / 4.0
pos_l0 = xpos - offset
pos_l1 = xpos + offset

for feat in features:
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))

    # -----------------------------
    # thresholds from HEALTHY-only fit
    # -----------------------------
    row = thr.loc[thr["feature"] == feat]
    if row.empty:
        raise ValueError(f"Feature '{feat}' not found in thr['feature'].")

    q = row.iloc[0]
    q_low  = float(q["q_low"])  if pd.notna(q.get("q_low", np.nan))  else np.nan
    q_high = float(q["q_high"]) if pd.notna(q.get("q_high", np.nan)) else np.nan
    rule   = FEATURE_RULES.get(feat, "two")

    # -----------------------------
    # data
    # -----------------------------
    healthy0 = pd.to_numeric(X_train[feat], errors="coerce").dropna()
    test0 = pd.to_numeric(Xte_scored.loc[Xte_scored["label"] == 0, feat], errors="coerce").dropna()
    test1 = pd.to_numeric(Xte_scored.loc[Xte_scored["label"] == 1, feat], errors="coerce").dropna()

    # label 0 boxes at both categories
    bp0 = ax.boxplot(
        [healthy0, test0],
        positions=pos_l0,
        widths=0.25,
        patch_artist=True,
        showfliers=True,
        whiskerprops=dict(linewidth=1.5, color="black"),
        capprops=dict(linewidth=1.5, color="black"),
        medianprops=dict(linewidth=1.5, color="black"),
    )

    for patch in bp0["boxes"]:
        patch.set_facecolor(label_colors[0])
        patch.set_alpha(0.85)

    # label 1 box only at Test (Healthy label=1 doesn't exist)
    if len(test1) > 0:
        bp1 = ax.boxplot(
            [test1],
            positions=[pos_l1[1]],
            widths=0.25,
            patch_artist=True,
            showfliers=True
        )
        for patch in bp1["boxes"]:
            patch.set_facecolor(label_colors[1])
            patch.set_alpha(0.85)

    # -----------------------------
    # y-limits: include thresholds + all data (including outliers)
    # -----------------------------
    combined = np.concatenate([healthy0.values, test0.values, test1.values]) if len(test1) else np.concatenate([healthy0.values, test0.values])
    
    data_min = combined.min()
    data_max = combined.max()

    candidates_max = [data_max]
    candidates_min = [data_min]
    
    if np.isfinite(q_low):
        candidates_max.append(q_low)
        candidates_min.append(q_low)
    if np.isfinite(q_high):
        candidates_max.append(q_high)
        candidates_min.append(q_high)

    ymax = max(candidates_max) * 1.10
    rng = data_max - data_min
    pad = 0.05 * rng if rng > 0 else 0.01  # fallback if constant

    ymin = min(candidates_min) - pad
    ymax = max(candidates_max) + pad

    ax.set_ylim(ymin, ymax)


    # -----------------------------
    # ✅ acceptance region shading (green) + threshold lines
    # -----------------------------
    if rule == "lower" and np.isfinite(q_low):
        ax.axhspan(q_low, ymax, facecolor="#2ca02c", alpha=0.10, zorder=0)
        ax.axhline(q_low, color="gray", linestyle="--", linewidth=1.2)

    elif rule == "upper" and np.isfinite(q_high):
        ax.axhspan(ymin, q_high, facecolor="#2ca02c", alpha=0.10, zorder=0)
        ax.axhline(q_high, color="gray", linestyle="--", linewidth=1.2)

    # cosmetics
    ax.set_xticks(xpos)
    ax.set_xticklabels(cats)
    ax.set_ylabel(feat, labelpad=15)
    ax.yaxis.set_label_coords(-0.10, 0.55)   # x-position, y-position

    ax.grid(axis="y", alpha=0.3)

    # legend
    legend_handles = [
        Patch(facecolor=label_colors[0], alpha=0.85, label="label = 0"),
        Patch(facecolor=label_colors[1], alpha=0.85, label="label = 1"),
    ]
    ax.legend(
        handles=legend_handles,
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=12,
        bbox_to_anchor=(0.5, 1.1)
    )

    fig.suptitle(
        f"Manual Brake Activation Fault Detection\n{feat}",
        fontsize=16,
        y=0.98
    )
    plt.tight_layout()
    plt.show()
    print("data_min:", data_min, "ymin:", ymin)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Ground truth
y_true = y_test.values  # 0=healthy, 1=fault

# Prediction from quantile-based detector
y_pred = Xte_scored["anomaly_flag_any"].values  # 0=normal, 1=anomaly

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("Confusion matrix (rows=true, cols=pred):\n", cm)

# Unpack
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Healthy", "Fault"]
))


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

plt.figure(figsize=(8, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred: Healthy", "Pred: Fault"],
    yticklabels=["True: Healthy", "True: Fault"]
)
plt.title("CM: Quantile-based \n Manual Brake Activation Detection", fontsize=20)
# plt.ylabel("True label",fontsize=18)
# plt.xlabel("Predicted label",fontsize=18)
plt.tight_layout()
plt.show()

# On wagon T3000 LP Kit 30

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
thresholds

imputer = SimpleImputer(strategy="median")

df_new = pd.read_csv("TestBrakefinal_data_raw_Dati30.csv")
df_new["label"] = df_new["LeakageLabel"].map({"Healthy": 0, "fault": 1})

cols_to_impute = [
    "First_phase_mean_curvature",
    "First_phase_half_time_ratio"
]

df_new[cols_to_impute] = imputer.fit_transform(df_new[cols_to_impute])

y_new = df_new["label"]
curv = pd.to_numeric(df_new["First_phase_mean_curvature"], errors="coerce")
htr  = pd.to_numeric(df_new["First_phase_half_time_ratio"], errors="coerce")

X_new_scored = df_new.copy()
X_new_scored["anomaly_flag"] = (
    (curv < thresholds["First_phase_mean_curvature"]["q_low"]) |
    (htr  > thresholds["First_phase_half_time_ratio"]["q_high"])
).fillna(False).astype(int)

cm = confusion_matrix(
    y_new,
    X_new_scored["anomaly_flag"],
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Healthy (0)", "Faulty (1)"],
    columns=["Pred Normal (0)", "Pred Anomaly (1)"]
)

print(cm_df)

